In [1]:
!pip install faiss-cpu #install FAISS

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/train.csv')

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.4 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [2]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

# Question 1

In [3]:
result = zs(prompt_150, candidate_labels=labels_150)

# Find the index of the ground-truth answer in the candidate labels
ans_index = labels_150.index(ans_150)

# Get the score
ground_truth_score = result['scores'][result['labels'].index(ans_150)]

print(f"The predicted probability score is: {round(ground_truth_score, 3)}")

The predicted probability score is: 0.384


# Question 2

In [4]:
query_embedding = model.encode([prompt_150], show_progress_bar=False)

# Query the FAISS index for the top 10 most similar documents
k = 10
distances, indices = index.search(query_embedding, k)

# The true correct document is kb[150]
true_correct_document_index = 150

# Find the rank of the true correct document in the retrieved indices
rank = -1
for i, idx in enumerate(indices[0]):
    if idx == true_correct_document_index:
        rank = i + 1  # Ranks are 1-based
        break

if rank != -1:
    print(f"The true correct document was placed at rank: {rank}")
else:
    print(f"The true correct document was not found in the top {k} retrieved documents.")

The true correct document was placed at rank: 10


# Question 3

In [5]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices[0]] #Get the top 10 chunks from the correct variable 'indices'
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

# Combine scores with their original indices from FAISS for re-ranking
scored_docs = list(zip(ce_scores, indices[0]))

# Sort by cross-encoder score in descending order
scored_docs.sort(key=lambda x: x[0], reverse=True)

# Find the rank of the true correct document in the re-ranked list
true_correct_document_index = 150
rank_ce = -1
for i, (score, doc_idx) in enumerate(scored_docs):
    if doc_idx == true_correct_document_index:
        rank_ce = i + 1
        break

if rank_ce != -1:
    print(f"The true correct document was placed at rank: {rank_ce} by the Cross-Encoder.")
else:
    print(f"The true correct document was not found in the top 10 documents after Cross-Encoder re-ranking.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

The true correct document was placed at rank: 1 by the Cross-Encoder.


# Question 4

In [6]:
# Get the prompt for row index 42
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# Embed the prompt
query_embedding_42 = model.encode([prompt_42], show_progress_bar=False)

# Query the FAISS index for the top k=5 most similar documents
k_q3 = 5
distances_q3, indices_q3 = index.search(query_embedding_42, k_q3)

# Retrieve the actual documents from the knowledge base
retrieved_docs_42 = [kb[idx] for idx in indices_q3[0]]

# Concatenate the documents with a single space
concatenated_docs = " ".join(retrieved_docs_42)

# Create the final string
input_string = f"Context: {concatenated_docs} Question: {prompt_42}"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the string without truncation
tokenized_input = tokenizer(input_string, truncation=False)

# Get the total number of tokens
total_tokens = len(tokenized_input['input_ids'])

print(f"The total number of tokens generated is: {total_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

The total number of tokens generated is: 216


# Question 5

In [7]:
# Retrieve the exact true document for row index 150 from the KB
true_document_150 = kb[150]

# Create the RAG string
rag_string_150 = f"Context: {true_document_150} Question: {prompt_150}"

# Run the zero-shot classification on this augmented string
result_rag = zs(rag_string_150, candidate_labels=labels_150)

# Get the score for the ground-truth answer from the RAG result
ground_truth_score_rag = result_rag['scores'][result_rag['labels'].index(ans_150)]

print(f"The new predicted score of the ground-truth correct option with RAG is: {round(ground_truth_score_rag, 3)}")

The new predicted score of the ground-truth correct option with RAG is: 0.989


# Question 6

In [8]:
# Retrieve the document at KB index 999
adversarial_document = kb[999]

# Create the Adversarial RAG string
adversarial_rag_string = f"Context: {adversarial_document} Question: {prompt_150}"

# Run the zero-shot classification on this adversarial string
result_adversarial_rag = zs(adversarial_rag_string, candidate_labels=labels_150)

# Get the score for the ground-truth answer from the adversarial RAG result
ground_truth_score_adversarial_rag = result_adversarial_rag['scores'][result_adversarial_rag['labels'].index(ans_150)]

print(f"The predicted probability score of the ground-truth correct option with Adversarial RAG is: {round(ground_truth_score_adversarial_rag, 3)}")

The predicted probability score of the ground-truth correct option with Adversarial RAG is: 0.529


# Question 7

In [9]:
hits = 0
total_prompts = 0
k_q7 = 5

for i in range(100):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    ans = str(row[row['answer']])

    # Embed the prompt
    query_embedding = model.encode([prompt], show_progress_bar=False)

    # Query the FAISS index for the top k=5 most similar documents
    distances, indices = index.search(query_embedding, k_q7)

    # Retrieve the actual documents from the knowledge base
    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check if the correct answer string is in any of the retrieved documents
    is_hit = False
    for doc in retrieved_docs:
        if ans in doc:
            is_hit = True
            break

    if is_hit:
        hits += 1
    total_prompts += 1

# Calculate the hit rate percentage
hit_rate_percentage = (hits / total_prompts) * 100

print(f"The exact Hit Rate percentage is: {round(hit_rate_percentage, 1)}%")

The exact Hit Rate percentage is: 73.0%


In [10]:
map_scores = []
k_retrieve = 5 # for FAISS retrieval
k_map = 3    # for MAP@3 calculation

for i in range(20): # First 20 rows (indices 0-19)
    row = train.iloc[i]
    prompt = str(row['prompt'])

    # Get the candidate labels (A, B, C, D, E) for the current row
    candidate_labels = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    ground_truth_answer_text = str(row[row['answer']]) # The actual text of the correct answer

    # Retrieve
    query_embedding = model.encode([prompt], show_progress_bar=False)
    distances, faiss_indices = index.search(query_embedding, k_retrieve)
    retrieved_docs = [kb[idx] for idx in faiss_indices[0]]

    # Rerank
    pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)

    # Find the index of the document with the highest cross-encoder score
    best_doc_idx_in_retrieved = np.argmax(ce_scores)
    best_document = retrieved_docs[best_doc_idx_in_retrieved]

    # Augment
    rag_string = f"Context: {best_document} Question: {prompt}"

    # Predict
    zs_result = zs(rag_string, candidate_labels=candidate_labels)

    predicted_labels_text_sorted = zs_result['labels']

    # Score
    ap_at_3 = 0.0
    for rank_idx, predicted_text in enumerate(predicted_labels_text_sorted[:k_map]):
        if predicted_text == ground_truth_answer_text:
            ap_at_3 = 1.0 / (rank_idx + 1)
            break

    map_scores.append(ap_at_3)

# Calculate the final average MAP@3
final_map_at_3 = np.mean(map_scores)

print(f"The final average MAP@3 score across the first 20 rows is: {round(final_map_at_3, 3)}")

The final average MAP@3 score across the first 20 rows is: 0.975
